In [1]:
library("xgboost")
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [2]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [3]:
data=read.csv('Hourly_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [4]:
dim(data)

[1] 414  27

In [5]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,24,1,24,0.8512223,4.041507e-09,5.3506983,-3.435885,0.8957711,2.150024,0.9766084,⋯,0.09813614,0.1914053,0.8944637,700,0.02403760,0.02403760,0.02403760,0,0,0
2,24,1,24,0.9194922,2.449422e-09,8.2809591,-1.146953,0.9141691,2.554141,0.9728383,⋯,0.08323323,0.1945535,0.9155878,700,0.02337670,0.02337670,0.02337670,0,0,0
3,24,1,24,0.9079559,3.241234e-09,-9.0028041,1.658264,0.9086270,2.440903,0.9656292,⋯,0.05203444,0.1276246,0.8720262,700,0.02334857,0.02334857,0.02334857,0,0,0
4,24,1,24,0.8712692,2.010702e-09,0.7677543,2.701202,0.9187827,2.040658,0.9830436,⋯,0.44839902,0.7087884,0.9048109,700,0.02348685,0.02348685,0.02348685,0,0,0
5,24,1,24,0.8801393,3.844576e-09,-0.0186239,5.952095,0.9412170,2.951870,0.9634139,⋯,0.45128188,0.6560024,0.8393581,700,0.02346539,0.02346539,0.02346539,0,0,0
6,24,1,24,0.8551463,3.051179e-09,-2.8143809,2.642106,0.8964147,2.388106,0.9742814,⋯,0.18178351,0.2069934,0.8804445,700,0.06628799,0.06628799,0.06628799,0,0,0


In [6]:
dlist= load('Hourly_nnetar_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [7]:
dim(MASE)

[1] 414   5   4

In [8]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [9]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [10]:
nanum

NULL

In [11]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [12]:
table(realbestmin)

realbestmin
  0   1   2   3   4 
100  66  59  70 119 

In [13]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [14]:
realbestmean

2
1
1
4
1
0
4
1
1
0
4


In [15]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,24,1,24,0.8512223,4.041507e-09,5.3506983,-3.435885,0.8957711,2.150024,0.9766084,⋯,0.09813614,0.1914053,0.8944637,700,0.02403760,0.02403760,0.02403760,0,0,0
2,24,1,24,0.9194922,2.449422e-09,8.2809591,-1.146953,0.9141691,2.554141,0.9728383,⋯,0.08323323,0.1945535,0.9155878,700,0.02337670,0.02337670,0.02337670,0,0,0
3,24,1,24,0.9079559,3.241234e-09,-9.0028041,1.658264,0.9086270,2.440903,0.9656292,⋯,0.05203444,0.1276246,0.8720262,700,0.02334857,0.02334857,0.02334857,0,0,0
4,24,1,24,0.8712692,2.010702e-09,0.7677543,2.701202,0.9187827,2.040658,0.9830436,⋯,0.44839902,0.7087884,0.9048109,700,0.02348685,0.02348685,0.02348685,0,0,0
5,24,1,24,0.8801393,3.844576e-09,-0.0186239,5.952095,0.9412170,2.951870,0.9634139,⋯,0.45128188,0.6560024,0.8393581,700,0.02346539,0.02346539,0.02346539,0,0,0
6,24,1,24,0.8551463,3.051179e-09,-2.8143809,2.642106,0.8964147,2.388106,0.9742814,⋯,0.18178351,0.2069934,0.8804445,700,0.06628799,0.06628799,0.06628799,0,0,0


In [16]:
set.seed(100)
index = sample(2,nrow(data),replace = TRUE,prob=c(0.7,0.3))

In [17]:
train_data=data[index==1,]
test_data=data[index==2,]
train_label_min=realbestmin[index==1,]
test_label_min=realbestmin[index==2,]
train_label_mean=realbestmean[index==1,]
test_label_mean=realbestmean[index==2,]

In [18]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,24,1,24,0.8512223,4.041507e-09,5.3506983,-3.435885,0.8957711,2.150024,0.9766084,⋯,0.09813614,0.1914053,0.8944637,700,0.02403760,0.02403760,0.02403760,0,0,0
2,24,1,24,0.9194922,2.449422e-09,8.2809591,-1.146953,0.9141691,2.554141,0.9728383,⋯,0.08323323,0.1945535,0.9155878,700,0.02337670,0.02337670,0.02337670,0,0,0
3,24,1,24,0.9079559,3.241234e-09,-9.0028041,1.658264,0.9086270,2.440903,0.9656292,⋯,0.05203444,0.1276246,0.8720262,700,0.02334857,0.02334857,0.02334857,0,0,0
4,24,1,24,0.8712692,2.010702e-09,0.7677543,2.701202,0.9187827,2.040658,0.9830436,⋯,0.44839902,0.7087884,0.9048109,700,0.02348685,0.02348685,0.02348685,0,0,0
5,24,1,24,0.8801393,3.844576e-09,-0.0186239,5.952095,0.9412170,2.951870,0.9634139,⋯,0.45128188,0.6560024,0.8393581,700,0.02346539,0.02346539,0.02346539,0,0,0
6,24,1,24,0.8551463,3.051179e-09,-2.8143809,2.642106,0.8964147,2.388106,0.9742814,⋯,0.18178351,0.2069934,0.8804445,700,0.06628799,0.06628799,0.06628799,0,0,0


In [19]:
end_time = Sys.time()

In [20]:
time_matrix[1,]=end_time-start_time

In [21]:
end_time-start_time

Time difference of 7.430921 secs

## Target the interval where the actual error is minimum

In [22]:
start_time = Sys.time()

In [23]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [24]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [25]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=100)

[1]	train-rmse:1.754079 
[2]	train-rmse:1.426518 
[3]	train-rmse:1.189110 
[4]	train-rmse:1.028226 
[5]	train-rmse:0.915751 
[6]	train-rmse:0.836870 
[7]	train-rmse:0.759872 
[8]	train-rmse:0.694982 
[9]	train-rmse:0.640896 
[10]	train-rmse:0.599680 
[11]	train-rmse:0.537154 
[12]	train-rmse:0.475537 
[13]	train-rmse:0.427245 
[14]	train-rmse:0.394142 
[15]	train-rmse:0.365099 
[16]	train-rmse:0.328297 
[17]	train-rmse:0.309963 
[18]	train-rmse:0.299233 
[19]	train-rmse:0.266722 
[20]	train-rmse:0.253993 
[21]	train-rmse:0.241670 
[22]	train-rmse:0.220850 
[23]	train-rmse:0.209880 
[24]	train-rmse:0.201576 
[25]	train-rmse:0.190691 
[26]	train-rmse:0.182536 
[27]	train-rmse:0.172621 
[28]	train-rmse:0.162848 
[29]	train-rmse:0.149969 
[30]	train-rmse:0.143698 
[31]	train-rmse:0.126855 
[32]	train-rmse:0.118047 
[33]	train-rmse:0.114683 
[34]	train-rmse:0.110180 
[35]	train-rmse:0.101690 
[36]	train-rmse:0.096500 
[37]	train-rmse:0.090840 
[38]	train-rmse:0.085306 
[39]	train-rmse:0.079

In [26]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.336927 
[2]	train-mlogloss:1.108247 
[3]	train-mlogloss:0.945720 
[4]	train-mlogloss:0.800212 
[5]	train-mlogloss:0.695897 
[6]	train-mlogloss:0.610626 
[7]	train-mlogloss:0.539893 
[8]	train-mlogloss:0.477578 
[9]	train-mlogloss:0.422898 
[10]	train-mlogloss:0.379453 
[11]	train-mlogloss:0.347821 
[12]	train-mlogloss:0.318073 
[13]	train-mlogloss:0.291677 
[14]	train-mlogloss:0.269218 
[15]	train-mlogloss:0.246646 
[16]	train-mlogloss:0.232475 
[17]	train-mlogloss:0.216141 
[18]	train-mlogloss:0.201310 
[19]	train-mlogloss:0.186213 
[20]	train-mlogloss:0.173002 
[21]	train-mlogloss:0.162077 
[22]	train-mlogloss:0.151775 
[23]	train-mlogloss:0.142280 
[24]	train-mlogloss:0.132913 
[25]	train-mlogloss:0.124454 
[26]	train-mlogloss:0.117341 
[27]	train-mlogloss:0.111062 
[28]	train-mlogloss:0.105443 
[29]	train-mlogloss:0.100259 
[30]	train-mlogloss:0.094415 
[31]	train-mlogloss:0.089462 
[32]	train-mlogloss:0.085561 
[33]	train-mlogloss:0.081516 
[34]	train-mlogloss

In [27]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1809
[LightGBM] [Info] Number of data points in the train set: 293, number of used features: 21
[LightGBM] [Info] Start training from score 2.109215
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [28]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 100,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000592 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1809
[LightGBM] [Info] Number of data points in the train set: 293, number of used features: 21
[LightGBM] [Info] Start training from score -1.431677
[LightGBM] [Info] Start training from score -1.851531
[LightGBM] [Info] Start training from score -1.895983
[LightGBM] [Info] Start training from score -1.808972
[LightGBM] [Info] Start training from score -1.237521
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [29]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [30]:
start_time = Sys.time()

In [31]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [32]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [33]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=100)

[1]	train-rmse:1.549655 
[2]	train-rmse:1.249129 
[3]	train-rmse:1.061063 
[4]	train-rmse:0.933098 
[5]	train-rmse:0.813894 
[6]	train-rmse:0.762802 
[7]	train-rmse:0.674252 
[8]	train-rmse:0.622831 
[9]	train-rmse:0.572754 
[10]	train-rmse:0.552619 
[11]	train-rmse:0.503704 
[12]	train-rmse:0.472615 
[13]	train-rmse:0.430372 
[14]	train-rmse:0.386408 
[15]	train-rmse:0.341767 
[16]	train-rmse:0.331640 
[17]	train-rmse:0.317406 
[18]	train-rmse:0.285973 
[19]	train-rmse:0.265783 
[20]	train-rmse:0.233876 
[21]	train-rmse:0.210831 
[22]	train-rmse:0.195281 
[23]	train-rmse:0.181433 
[24]	train-rmse:0.175499 
[25]	train-rmse:0.166980 
[26]	train-rmse:0.154792 
[27]	train-rmse:0.145108 
[28]	train-rmse:0.136190 
[29]	train-rmse:0.130952 
[30]	train-rmse:0.119489 
[31]	train-rmse:0.112317 
[32]	train-rmse:0.107099 
[33]	train-rmse:0.095624 
[34]	train-rmse:0.092639 
[35]	train-rmse:0.083314 
[36]	train-rmse:0.076193 
[37]	train-rmse:0.073675 
[38]	train-rmse:0.070546 
[39]	train-rmse:0.065

In [34]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.293444 
[2]	train-mlogloss:1.065831 
[3]	train-mlogloss:0.904407 
[4]	train-mlogloss:0.776237 
[5]	train-mlogloss:0.674723 
[6]	train-mlogloss:0.577363 
[7]	train-mlogloss:0.515430 
[8]	train-mlogloss:0.448781 
[9]	train-mlogloss:0.410910 
[10]	train-mlogloss:0.363751 
[11]	train-mlogloss:0.322460 
[12]	train-mlogloss:0.294463 
[13]	train-mlogloss:0.266970 
[14]	train-mlogloss:0.244399 
[15]	train-mlogloss:0.225562 
[16]	train-mlogloss:0.210434 
[17]	train-mlogloss:0.196210 
[18]	train-mlogloss:0.179992 
[19]	train-mlogloss:0.170812 
[20]	train-mlogloss:0.160558 
[21]	train-mlogloss:0.150853 
[22]	train-mlogloss:0.139821 
[23]	train-mlogloss:0.130379 
[24]	train-mlogloss:0.121582 
[25]	train-mlogloss:0.114451 
[26]	train-mlogloss:0.107585 
[27]	train-mlogloss:0.101813 
[28]	train-mlogloss:0.095875 
[29]	train-mlogloss:0.091049 
[30]	train-mlogloss:0.086486 
[31]	train-mlogloss:0.082573 
[32]	train-mlogloss:0.079254 
[33]	train-mlogloss:0.075730 
[34]	train-mlogloss

In [35]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005573 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1809
[LightGBM] [Info] Number of data points in the train set: 293, number of used features: 21
[LightGBM] [Info] Start training from score 1.767918
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [36]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000485 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1809
[LightGBM] [Info] Number of data points in the train set: 293, number of used features: 21
[LightGBM] [Info] Start training from score -1.202836
[LightGBM] [Info] Start training from score -1.709881
[LightGBM] [Info] Start training from score -1.873510
[LightGBM] [Info] Start training from score -1.709881
[LightGBM] [Info] Start training from score -1.691189
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [37]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [38]:
end_time-start_time

Time difference of 33.32866 secs

## predict

In [39]:
start_time = Sys.time()

In [40]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [41]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [42]:
xgbregmin

[1]  1.998953e+00  2.999349e+00  1.000835e+00  1.476962e-04  1.001425e+00
  [6]  2.142525e-04  3.801980e+00  3.998814e+00  2.633948e-04  3.999939e+00
 [11]  4.000394e+00  2.646631e+00  3.999354e+00 -1.501857e-04  2.602381e+00
 [16]  4.311104e-03  1.002409e+00  1.001480e+00 -4.136703e-04  9.995874e-01
 [21]  1.997759e+00  8.213826e-01  1.286534e-03  1.027591e+00  3.999042e+00
 [26]  1.471940e-03  2.569144e+00  3.937135e+00  1.999449e+00  3.996399e+00
 [31]  2.996437e+00  1.788467e+00  1.002579e+00  1.691461e+00  1.000002e+00
 [36]  2.471199e+00  3.999868e+00  4.000184e+00  2.817363e+00  2.183351e-03
 [41]  4.000184e+00  1.999930e+00  1.329466e+00  2.204340e+00  1.115220e-03
 [46] -6.183041e-04 -1.712011e-01  1.537667e+00  3.998386e+00  9.998744e-01
 [51]  2.000315e+00  6.219127e-04  1.000221e+00  3.998654e+00  3.999416e+00
 [56]  1.718793e-03  4.001326e+00 -5.729318e-04  1.226709e-03  2.995728e+00
 [61]  2.000329e+00  1.000388e+00  2.176264e+00  3.001295e+00  1.999948e+00
 [66]  3.998907e+00  4.000719e+00  1.741615e-03  2.999228e+00  3.999164e+00
 [71]  3.998935e+00  1.006065e+00  4.000143e+00  2.782937e-01  1.000560e+00
 [76]  4.678030e-04  2.017277e+00  9.674459e-01  1.234645e+00  1.998316e+00
 [81]  7.448808e-04  1.999819e+00  2.552356e+00  1.724539e+00  9.998212e-01
 [86]  3.998655e+00  2.386220e+00  2.052605e-05 -3.617051e-04  3.649397e+00
 [91]  2.801048e+00  4.000034e+00  3.667758e-04  4.111812e-05  1.081294e+00
 [96]  3.999906e+00  3.999214e+00  2.000227e+00  3.998159e+00  2.696720e+00
[101]  8.948586e-04  4.000310e+00  7.635558e-04  1.999672e+00  9.996099e-01
[106]  2.999702e+00  2.102261e+00  2.318580e+00  2.001211e+00  2.995038e+00
[111]  1.460030e+00  2.461166e+00  2.000387e+00  8.908738e-05  3.484048e+00
[116]  2.000407e+00  9.771343e-01  2.730732e+00  3.997576e+00  7.275441e-01
[121]  3.998194e+00  3.999424e+00  2.800769e-03  2.002748e+00  2.994889e+00
[126]  3.000400e+00  1.915647e+00  7.053884e-01  1.999557e+00  9.984160e-05
[131] -7.093707e-05  3.442372e-01  9.842417e-01  3.999830e+00  9.532446e-04
[136]  9.999562e-01 -1.436000e-04  2.996813e+00  2.001467e+00  1.001444e+00
[141]  1.000024e+00  3.866310e-03  3.997362e+00  2.556392e+00  1.000199e+00
[146]  2.959180e+00  2.548652e+00  2.997536e+00  1.000708e+00  4.189939e-04
[151]  4.000589e+00  2.286939e+00  6.254422e-04  4.000319e+00  9.968333e-01
[156]  2.411686e+00 -2.411197e-02  1.978404e+00  1.999962e+00  3.999954e+00
[161]  9.283929e-01  3.999030e+00  1.000297e+00  1.001132e+00  2.999061e+00
[166]  1.001498e+00  9.997318e-01  2.781759e+00  2.416405e+00  3.854815e+00
[171]  3.000483e+00  2.000311e+00  2.000109e+00  2.451133e+00  2.580112e+00
[176]  1.000450e+00  3.339030e+00  3.999590e+00  1.003090e+00  3.999189e+00
[181]  3.995978e+00  2.290852e+00  3.253838e+00  3.001688e+00  3.999504e+00
[186]  2.000325e+00  2.040403e-03  3.997955e+00  2.000402e+00  6.215892e-01
[191]  2.819391e+00  3.000044e+00  1.248154e-03  1.283756e+00  3.000309e+00
[196]  3.994802e+00  3.000912e+00  1.258876e-03  2.999655e+00  1.765737e+00
[201]  3.999521e+00  2.756090e+00  8.299021e-01  2.757907e+00  2.000922e+00
[206]  8.488621e-01  2.033750e+00  3.997979e+00  2.999764e+00  2.002242e+00
[211]  1.999928e+00  2.819696e+00  1.280841e-03  3.000302e+00  2.999138e+00
[216]  3.998133e+00  2.692478e+00  1.916916e+00  4.000061e+00  1.999247e+00
[221]  3.377972e-03  3.999620e+00  2.000426e+00  1.638760e+00  2.785912e-03
[226]  3.998649e+00  3.999435e+00  3.999521e+00  2.000147e+00  2.999754e+00
[231]  4.000305e+00  3.998685e+00  1.685941e+00  2.999717e+00  2.003718e+00
[236]  4.000456e+00  3.999589e+00  6.383260e-01  2.179715e+00  1.294742e+00
[241]  3.000242e+00  2.319597e+00  2.999777e+00  3.082466e+00  1.999232e+00
[246]  2.443276e+00  3.000026e+00  2.819781e-03  3.000189e+00  2.000051e+00
[251]  2.999789e+00  2.542407e+00  2.999390e+00 -6.680128e-04  2.583658e+00
[256]  1.421138e+00  3.996334e+00 -6.364976e-05  5.788520e-03  2.239192e+00
[261]  3.999755e+00  8.542349e-01  1.000522e+00  1

In [43]:
xgbclsmin

[1] 2 3 1 0 1 0 4 4 0 4 4 0 4 0 0 0 1 1 0 1 2 0 0 0 4 0 2 4 2 4 3 0 1 2 1 4 4
 [38] 4 4 0 4 3 0 4 0 0 0 0 4 1 2 0 1 4 4 0 4 0 0 3 2 1 4 3 2 4 4 0 3 4 4 1 4 0
 [75] 1 0 0 0 1 2 0 2 4 4 1 4 1 0 0 4 2 4 0 0 0 4 4 2 4 4 0 4 0 2 1 3 0 4 2 3 4
[112] 2 2 0 4 2 1 4 4 0 4 4 0 2 3 3 3 4 2 0 0 1 1 4 0 1 0 3 2 1 1 0 4 0 1 1 1 3
[149] 1 0 4 1 0 4 1 4 0 4 2 4 0 4 1 1 3 1 1 1 1 4 3 2 2 3 4 1 4 4 1 4 4 3 2 3 4
[186] 2 0 4 2 0 4 3 0 0 3 4 3 0 3 4 4 4 1 4 2 4 4 4 3 2 2 4 0 3 3 4 4 0 4 2 0 4
[223] 2 0 0 4 4 4 2 3 4 4 4 3 2 4 4 4 4 2 3 4 3 2 2 4 3 0 3 2 3 4 3 0 4 0 4 0 0
[260] 4 4 4 1 3 0 3 2 4 0 4 4 4 3 3 3 3 4 4 3 3 2 4 3 0 2 2 4 2 2 2 1 3 4 3 2 4
[297] 4 4 3 4 2 4 0 0 4 4 4 4 4 4 4 2 0 1 4 0 0 0 3 4 4 0 1 2 1 4 4 4 0 1 0 1 4
[334] 0 0 1 4 0 1 0 0 0 4 1 3 3 1 2 4 0 3 0 4 1 1 4 0 2 1 2 0 1 4 0 3 4 2 2 4 1
[371] 3 1 0 4 1 1 1 3 1 1 0 4 3 0 1 1 0 0 4 3 1 0 0 0 2 3 4 0 1 4 4 2 4 0 2 4 2
[408] 0 0 3 0 0 1 1

In [44]:
lgbregmin

[1]  1.63809744  2.32771357  1.36400478  1.00543851  1.94706120  0.15343105
  [7]  2.82095115  3.24705965  0.27475332  3.90180095  3.64539181  1.99009424
 [13]  3.77994744  0.17416126  1.94063290  0.17705145  1.76388750  1.21227029
 [19]  0.37598153  1.11120381  1.76038995  0.38101898  0.72627740  0.99338596
 [25]  3.50923313  0.53321066  1.92925696  3.19130971  1.99803927  3.36882198
 [31]  2.65394967  1.80536537  1.67904903  1.91151736  1.54713811  2.13753990
 [37]  3.77891334  3.70135348  2.82612771  0.29989378  3.67056980  3.26788821
 [43]  1.77081089  2.74669508  0.46341255  0.04857068  0.85106445  1.21028066
 [49]  3.23069355  0.25517152  1.77241726  0.15762195  1.37532392  3.62482177
 [55]  3.72987647  1.42900968  3.67605729 -0.01812686  0.48463232  2.61257266
 [61]  2.05966082  1.50138851  3.08390658  3.06371277  2.43780078  3.31374168
 [67]  3.32184840 -0.02660186  2.65035991  3.60411778  3.68573594  1.02593404
 [73]  3.96322313  1.31085908  1.51743409  1.04521068  2.87086013  1.20032505
 [79]  1.77727622  1.56731297 -0.30745294  2.15041605  1.85347133  1.50214888
 [85]  1.35621147  3.36652811  1.85193880  0.92761770  0.34087308  3.39681529
 [91]  1.95409143  3.48388371  0.59557756  0.87085713  0.84719197  3.57496784
 [97]  3.75125328  1.65271790  3.93977471  2.13665518  0.63805263  3.63588084
[103]  0.60544911  2.06869427  0.78934136  2.97947765  1.21007570  3.10452407
[109]  1.96042773  2.33452497  1.38904457  2.10911399  1.56749937 -0.16399295
[115]  2.44714864  2.00879635  0.97243320  3.16495118  3.65109624  1.49530172
[121]  4.06074646  3.86429298  0.64382073  2.07080665  2.35977616  2.71878509
[127]  2.09309482  2.06503374  1.48870510  0.22032960  0.28978425  0.95053314
[133]  1.52582279  3.17348096  1.25692170  1.25756503  0.40156553  2.66734826
[139]  2.01028776  1.50103035  1.38494692  0.24506347  2.79618958  1.52307829
[145]  0.92252520  2.05419620  1.65149652  3.01977972  1.12488588  0.53232125
[151]  3.87555580  2.00087789  0.48649261  3.75890275  0.74382306  1.82167203
[157]  1.05731492  1.20837093  1.67467043  3.44827860  0.95414339  3.37881157
[163]  1.63717813  1.62531117  2.36637464  1.44297893  1.39629902  2.34655013
[169]  2.24419905  3.64154872  3.00751045  1.97471638  2.25764786  1.84977606
[175]  2.34232098  1.68167851  3.83392921  3.37494255  1.31514205  3.00013212
[181]  3.19454880  2.70429567  2.66466873  2.78433986  3.46090099  1.72884389
[187]  0.93412066  3.66785796  1.54381825  0.47693521  2.31198778  3.24753149
[193]  0.71498382  0.55670448  2.86385113  3.27093427  3.04025243  0.92129485
[199]  3.23795077  1.55291354  4.07552430  2.25354002  1.54557312  3.42703437
[205]  2.06989721  2.17627955  1.96394343  3.75225326  2.48555299  2.78328092
[211]  2.17018612  3.30928990  0.40461239  3.05439444  2.57419803  3.13366270
[217]  2.66285555  2.02346091  3.92895048  2.05543812  0.78974612  3.66053229
[223]  2.45505093  0.95252746  0.63314831  3.92473770  3.61648372  3.55128325
[229]  2.22277724  2.79702770  3.78370899  3.28425023  2.31899655  3.09909115
[235]  2.36613559  3.82911883  3.90007433  2.44606505  3.01061634  2.26950548
[241]  2.49783412  3.03994384  3.03597410  2.64443597  1.54228803  3.61686701
[247]  2.97502916  0.96971690  2.66891915  2.35275645  3.02048807  3.29437752
[253]  2.70361802  0.52365888  2.13038877  2.32534676  3.31456620  0.22832062
[259]  1.27385188  2.94899853  3.35275935  1.96244772  1.53600848  2.38109867
[265]  0.15008163  2.93274147  2.01565065  3.24910612  1.85484839  3.19661246
[271]  3.81917714  2.47198421  3.48339290  3.05327032  3.42955104  2.46055837
[277]  4.09880486  3.39967620  2.83944237  2.99071678  1.94596954  3.91005929
[283]  3.18185838  1.08302036  2.05556498  2.40358507  2.51913834  1.29890895
[289]  2.20687916  1.88331843  1.26699138  3.18996910  3.58958414  2.96961049
[295]  1.40251690  3.79507693  3.42441818  2.02994172  2.58455484  3.71453972
[301]  2.00022762  3.58543450  1.05426659  0.78766007  3.55023650  3.62860513
[307]  3.88028797  3.801

In [45]:
lgbclsmin

0.014380798,0.007974292,9.755254e-01,1.057739e-03,0.001061818
0.016073760,0.009727942,7.785087e-03,9.636115e-01,0.002801716
0.010025921,0.968030854,4.056379e-04,1.286385e-03,0.020251203
0.898859719,0.038971530,2.838747e-03,6.755433e-03,0.052574572
0.008423550,0.937124017,7.374554e-04,2.184170e-03,0.051530807
0.971987189,0.013720894,6.462716e-04,4.117012e-03,0.009528633
0.005780626,0.001005591,1.630048e-04,2.679546e-04,0.992782823
0.011550861,0.015445538,7.315176e-04,4.603927e-03,0.967668157
0.951609803,0.029819191,2.393692e-04,1.640656e-03,0.016690981
0.005697552,0.007757414,2.387594e-03,2.055974e-03,0.982101466
0.014335447,0.002936571,7.177054e-04,6.137923e-04,0.981396483


In [46]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [47]:
lgbclsminr

3
0
3
3
2
0
1
4
2
0
1


In [48]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [49]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [50]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [51]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
2,1.9989531040,3,1.638097
3,2.9993493557,0,2.327714
1,1.0008352995,3,1.364005
0,0.0001476962,3,1.005439
1,1.0014251471,2,1.947061
0,0.0002142525,0,0.153431


In [52]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [53]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
2,1.9998954535,2,1.6659382
1,1.0003926754,4,1.2679726
1,0.9987677336,3,1.2723569
4,3.9979720116,3,3.5055921
1,1.0018657446,4,2.5799506
0,-0.0006571151,2,0.1683413


In [54]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
2,1.9998954535,2,1.6659382
1,1.0003926754,4,1.2679726
1,0.9987677336,3,1.2723569
4,3.9979720116,3,3.5055921
1,1.0018657446,4,2.5799506
0,-0.0006571151,2,0.1683413


In [55]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [56]:
result_list=list(preallmin,preallmean,time_matrix)

In [60]:
save(result_list, file = "Hourly_nnetar_opt_pre_result.RData")

In [58]:
time_matrix

user_time,system_time,elapsed_time
7.430921,7.430921,7.430921
4.603225,4.603225,4.603225
33.328655,33.328655,33.328655
9.284281,9.284281,9.284281


In [59]:
importance_matrix1 <- xgb.importance(model = xgb_mean_cl)
importance_matrix2 <- xgb.importance(model = xgb_mean_reg)
importance_matrix3 <- lgb.importance(lgb_mean_cl)
importance_matrix4 <- lgb.importance(lgb_mean_reg)

importance_matrix1=importance_matrix1[order(importance_matrix1$Feature),]
importance_matrix2=importance_matrix2[order(importance_matrix2$Feature),]
importance_matrix3=importance_matrix3[order(importance_matrix3$Feature),]
importance_matrix4=importance_matrix4[order(importance_matrix4$Feature),]

importance_matrix_mean=(importance_matrix1[,2:4]+importance_matrix2[,2:4]+importance_matrix3[,2:4]+importance_matrix4[,2:4])/4

importance=cbind(importance_matrix1[,1],importance_matrix_mean)

write.csv(importance,'d_ets_imp_mean.csv')

ERROR: Error in Ops.data.frame(importance_matrix1[, 2:4] + importance_matrix2[, : ‘+’ only defined for equally-sized data frames
